# Kenya Cost of Living Analysis (2025–2026)

## What I wanted to understand

How are changing consumer prices and public transport costs affecting the monthly budget of a low-income household in Kenya?

For this project, I compare selected price observations for September 2025 and September 2026, build a small fixed-quantity grocery basket, and test how different commuting fares affect a sample household budget.

The household is an illustration to make the calculations easier to understand. It is not meant to represent every household in Kenya.

### Scenario used in this analysis
- Monthly household income: KSh 30,000
- Household: one adult and one child
- Accommodation: single room
- Commuting: public transport
- Working days used for the estimate: 22 per month
- Daily commute: two one-way journeys

### Data and tools
The raw price file is attributed to the Kenya National Bureau of Statistics (KNBS). I use Python, pandas and Matplotlib in Google Colab, with GitHub to keep the project files and documentation together.

**A note on interpretation:** The selected items are not the full official CPI basket. Some household expenses and all transport-fare scenarios are assumptions. I keep those limits visible throughout the notebook.

## 1. Load the price data

I load the CSV directly from the project repository so the notebook can be rerun without manually uploading the file. The raw dataset is kept unchanged; calculated columns are added to a working copy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_URL = (
    "https://raw.githubusercontent.com/Toxicnic/"
    "kenya-cost-of-living-2026/main/data/raw/knbs-prices.csv"
)

df = pd.read_csv(DATA_URL)

required_columns = {"item", "unit", "sep_2025", "sep_2026", "source"}
missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(f"The dataset is missing these columns: {sorted(missing_columns)}")

for column in ["sep_2025", "sep_2026"]:
    df[column] = pd.to_numeric(df[column], errors="raise")

print(f"Loaded {len(df)} price observations.")
display(df.head())

## 2. Compare price changes

For each item, I calculate the difference in shillings and the percentage change between the two observations. Sorting by percentage change makes the largest increases and decreases easier to spot.

In [ ]:
price_changes = df.copy()

price_changes["change_ksh"] = (
    price_changes["sep_2026"] - price_changes["sep_2025"]
).round(2)

price_changes["change_percent"] = (
    price_changes["change_ksh"] / price_changes["sep_2025"] * 100
).round(2)

price_changes = price_changes.sort_values(
    "change_percent", ascending=False
).reset_index(drop=True)

display(
    price_changes[
        ["item", "unit", "sep_2025", "sep_2026", "change_ksh", "change_percent"]
    ]
)

print(f"Items analysed: {len(price_changes)}")
print(f"Items with increases: {(price_changes['change_percent'] > 0).sum()}")
print(f"Items with decreases: {(price_changes['change_percent'] < 0).sum()}")
print(f"Items unchanged: {(price_changes['change_percent'] == 0).sum()}")

### What stands out?

The price movements are mixed. In this selected dataset, sukuma wiki, cabbage and tomatoes show some of the largest percentage increases, while sugar and fortified maize flour show decreases. These are changes in individual listed prices, not category-wide inflation rates.

## 3. Build one fixed-quantity grocery basket

To make the comparison more meaningful, I keep the selected quantities the same in both years. The quantities are illustrative choices for this analysis; they are not an official KNBS basket or a full monthly food plan.

This is the main grocery-basket calculation in the notebook. The duplicate version from the earlier draft has been removed.

In [ ]:
basket_quantities = {
    "Loose maize flour": 2.0,       # kg
    "Beans": 0.5,                   # kg
    "Sukuma wiki": 0.5,             # kg
    "Cabbage": 0.5,                 # kg
    "Tomatoes": 0.5,                # kg
    "Fresh packeted milk": 1.0,     # 500 ml packets
    "Eggs": 6.0,                    # pieces
    "White bread": 1.0,             # 400 g loaves
    "Sugar": 0.25,                  # kg
    "Cooking oil": 0.25,            # litres
}

price_lookup = df.set_index("item")
basket_rows = []

for item, quantity in basket_quantities.items():
    if item not in price_lookup.index:
        raise KeyError(f"Basket item not found in the raw data: {item}")

    row = price_lookup.loc[item]
    cost_2025 = quantity * row["sep_2025"]
    cost_2026 = quantity * row["sep_2026"]

    basket_rows.append({
        "item": item,
        "quantity_used": quantity,
        "unit": row["unit"],
        "cost_sep_2025_ksh": round(cost_2025, 2),
        "cost_sep_2026_ksh": round(cost_2026, 2),
        "change_ksh": round(cost_2026 - cost_2025, 2),
    })

basket_df = pd.DataFrame(basket_rows)

basket_2025 = basket_df["cost_sep_2025_ksh"].sum()
basket_2026 = basket_df["cost_sep_2026_ksh"].sum()
basket_difference = basket_2026 - basket_2025
basket_percent_change = basket_difference / basket_2025 * 100

display(basket_df)

print(f"September 2025 basket cost: KSh {basket_2025:,.2f}")
print(f"September 2026 basket cost: KSh {basket_2026:,.2f}")
print(f"Additional cost: KSh {basket_difference:,.2f}")
print(f"Percentage change: {basket_percent_change:.2f}%")
print(f"Balance from KSh 1,000 in 2025: KSh {1000 - basket_2025:,.2f}")
print(f"Balance from KSh 1,000 in 2026: KSh {1000 - basket_2026:,.2f}")

The fixed-quantity basket rises from **KSh 732.36** to **KSh 791.15**, an increase of **KSh 58.79 (8.03%)**. This result describes only the selected quantities and prices. It should not be read as a complete monthly food bill.

## 4. Keep additional reference prices separate

The following electricity and LPG observations are added to a separate working dataset for comparison. They do not alter the original CSV. The listed units (200 kWh and a 13 kg cylinder) are reference units, not estimates of what this household actually consumes each month.

In [ ]:
additional_prices = pd.DataFrame([
    {
        "item": "Electricity",
        "unit": "200 kWh",
        "sep_2025": 5597.16,
        "sep_2026": 5533.80,
        "source": "KNBS CPI Explorer"
    },
    {
        "item": "Gas/LPG",
        "unit": "13 kg cylinder",
        "sep_2025": 3151.65,
        "sep_2026": 3419.20,
        "source": "KNBS CPI Explorer"
    }
])

expanded_df = pd.concat([df.copy(), additional_prices], ignore_index=True)
expanded_df["change_ksh"] = (
    expanded_df["sep_2026"] - expanded_df["sep_2025"]
).round(2)
expanded_df["change_percent"] = (
    expanded_df["change_ksh"] / expanded_df["sep_2025"] * 100
).round(2)

display(
    expanded_df.tail(5)[
        ["item", "unit", "sep_2025", "sep_2026", "change_ksh", "change_percent", "source"]
    ]
)

## 5. Set up the household budget

I use KSh 30,000 as the scenario's monthly income. The rent figure is a reference price; most other budget lines are planning assumptions. They are included to explore how a budget behaves, not to claim that these are verified average household expenses.

In [ ]:
household = {
    "monthly_income_ksh": 30000,
    "adults": 1,
    "children": 1,
    "rent_type": "single room",
    "working_days_per_month": 22,
}

monthly_budget = {
    "Rent": 4235.00,
    "Food": 10000.00,                 # Illustrative planning estimate
    "Transport": 4400.00,             # KSh 100 each way for 22 working days
    "Cooking fuel": 800.00,           # Provisional estimate
    "Electricity": 500.00,            # Provisional estimate
    "Water": 300.00,                  # Provisional estimate
    "Phone and data": 1000.00,        # Provisional estimate
    "Child-related needs": 2000.00,   # Provisional estimate
    "Toiletries and cleaning": 1000.00,
    "Healthcare and emergencies": 1000.00,
}

monthly_income = household["monthly_income_ksh"]
total_expenses = sum(monthly_budget.values())
income_remaining = monthly_income - total_expenses

budget_df = pd.DataFrame(
    [{"category": category, "monthly_ksh": amount}
     for category, amount in monthly_budget.items()]
)

display(budget_df)

print(f"Total modelled expenses: KSh {total_expenses:,.2f}")
print(f"Monthly income: KSh {monthly_income:,.2f}")
print(f"Income remaining: KSh {income_remaining:,.2f}")
print(f"Share of income allocated to expenses: {total_expenses / monthly_income * 100:.1f}%")

### How to read this budget

Under these assumptions, the listed expenses use 84.1% of the KSh 30,000 income, leaving KSh 4,765. That balance depends on the assumptions above and is not a measured savings figure. The KSh 10,000 food line is separate from the small fixed-quantity basket.

## 6. Record where the budget figures came from

This audit table makes it easier to distinguish a reference price from a scenario or an estimate that still needs evidence.

In [ ]:
budget_audit = pd.DataFrame([
    ["Rent", 4235.00, "KNBS reference price", "National reference; local rent varies"],
    ["Food", 10000.00, "Assumption", "Illustrative monthly planning estimate"],
    ["Transport", 4400.00, "Scenario", "KSh 100 one-way x 2 trips x 22 days"],
    ["Cooking fuel", 800.00, "Assumption", "Needs validation against actual spending"],
    ["Electricity", 500.00, "Assumption", "Needs validation against actual spending"],
    ["Water", 300.00, "Assumption", "Needs validation against actual spending"],
    ["Phone and data", 1000.00, "Assumption", "Needs validation against actual spending"],
    ["Child-related needs", 2000.00, "Assumption", "Depends on the household"],
    ["Toiletries and cleaning", 1000.00, "Assumption", "Needs validation against actual spending"],
    ["Healthcare and emergencies", 1000.00, "Assumption", "Variable expense allowance"],
], columns=["category", "monthly_ksh", "evidence_status", "notes"])

display(budget_audit)
print(f"Budget audit total: KSh {budget_audit['monthly_ksh'].sum():,.2f}")

## 7. Test different public transport fares

I model four possible one-way fares: KSh 50, 100, 150 and 200. Each scenario assumes two one-way trips on each of 22 working days. The fares are hypothetical inputs, not verified average matatu fares.

For a fair comparison, all non-transport budget lines stay the same while the transport amount changes.

In [ ]:
fare_scenarios = [50, 100, 150, 200]
working_days = household["working_days_per_month"]
monthly_income = household["monthly_income_ksh"]

other_expenses = total_expenses - monthly_budget["Transport"]
scenario_rows = []

for fare in fare_scenarios:
    monthly_transport = fare * 2 * working_days
    scenario_expenses = other_expenses + monthly_transport
    remaining = monthly_income - scenario_expenses

    scenario_rows.append({
        "one_way_fare_ksh": fare,
        "monthly_transport_ksh": monthly_transport,
        "total_monthly_expenses_ksh": scenario_expenses,
        "income_remaining_ksh": remaining,
        "income_spent_percent": round(scenario_expenses / monthly_income * 100, 2),
    })

sensitivity_df = pd.DataFrame(scenario_rows)
display(sensitivity_df)

## 8. Visualise the transport scenarios

The chart shows how the amount remaining changes when the one-way fare changes, with the other budget assumptions held constant.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

bars = ax.bar(
    sensitivity_df["one_way_fare_ksh"].astype(str),
    sensitivity_df["income_remaining_ksh"]
)

ax.set_title("Illustrative Transport Fares and Monthly Income Remaining")
ax.set_xlabel("Assumed one-way fare (KSh)")
ax.set_ylabel("Income remaining after modelled expenses (KSh)")
ax.axhline(0, linewidth=0.8)

for bar, amount in zip(bars, sensitivity_df["income_remaining_ksh"]):
    ax.annotate(
        f"KSh {amount:,.0f}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 5 if amount >= 0 else -15),
        textcoords="offset points",
        ha="center",
        fontsize=9,
    )

fig.text(
    0.5, 0.01,
    "Illustrative scenarios only. Non-transport budget assumptions are held constant.",
    ha="center",
    fontsize=9,
)
fig.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

### What the scenarios suggest

At a KSh 50 one-way fare, the model leaves KSh 6,965 after the listed expenses. At KSh 200, it leaves KSh 365. This is not a prediction of what any specific household will spend; it shows how sensitive this particular budget is to the fare assumption.

## 9. Audit the calculations

I use independent calculations and assertions to catch accidental changes to the key results. These checks confirm arithmetic consistency; they do not verify the original source observations or the realism of the assumptions.

In [ ]:
# Recalculate the price changes from the raw columns.
audit_df = df.copy()
audit_df["audit_change_ksh"] = (audit_df["sep_2026"] - audit_df["sep_2025"]).round(2)
audit_df["audit_change_percent"] = (
    audit_df["audit_change_ksh"] / audit_df["sep_2025"] * 100
).round(2)

assert len(df) == 16, "The raw dataset row count has changed."
assert np.allclose(
    audit_df["audit_change_ksh"],
    (df["sep_2026"] - df["sep_2025"]).round(2)
)
assert round(basket_2025, 2) == 732.36
assert round(basket_2026, 2) == 791.15
assert round(basket_difference, 2) == 58.79
assert round(basket_percent_change, 2) == 8.03
assert round(total_expenses, 2) == 25235.00
assert round(income_remaining, 2) == 4765.00
assert budget_audit["monthly_ksh"].sum() == total_expenses

expected_transport = (
    sensitivity_df["one_way_fare_ksh"] * 2 * working_days
)
assert (expected_transport == sensitivity_df["monthly_transport_ksh"]).all()
assert (
    sensitivity_df["income_remaining_ksh"]
    == monthly_income - sensitivity_df["total_monthly_expenses_ksh"]
).all()

print("All calculation checks passed.")
print(f"Raw price observations audited: {len(audit_df)}")
print(f"Fixed-quantity basket increase: {basket_percent_change:.2f}%")
print(f"Illustrative monthly budget: KSh {total_expenses:,.2f}")
print(f"Remaining income at KSh 100 one-way fare: KSh {income_remaining:,.2f}")

## 10. Save the project outputs

The following files are created for the repository. In Google Colab, they appear in the Files panel and can be downloaded or uploaded to GitHub.

In [ ]:
output_dir = Path("processed")
visual_dir = Path("visuals")
output_dir.mkdir(exist_ok=True)
visual_dir.mkdir(exist_ok=True)

price_changes.to_csv(output_dir / "price_changes.csv", index=False)
basket_df.to_csv(output_dir / "fixed_grocery_basket.csv", index=False)
budget_audit.to_csv(output_dir / "household_budget_audit.csv", index=False)
sensitivity_df.to_csv(output_dir / "transport_sensitivity_analysis.csv", index=False)

# Save the chart from the current figure by creating it once more for export.
fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.bar(
    sensitivity_df["one_way_fare_ksh"].astype(str),
    sensitivity_df["income_remaining_ksh"]
)
ax.set_title("Illustrative Transport Fares and Monthly Income Remaining")
ax.set_xlabel("Assumed one-way fare (KSh)")
ax.set_ylabel("Income remaining after modelled expenses (KSh)")
ax.axhline(0, linewidth=0.8)
for bar, amount in zip(bars, sensitivity_df["income_remaining_ksh"]):
    ax.annotate(
        f"KSh {amount:,.0f}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 5 if amount >= 0 else -15),
        textcoords="offset points",
        ha="center",
        fontsize=9,
    )
fig.text(
    0.5, 0.01,
    "Illustrative scenarios only. Non-transport budget assumptions are held constant.",
    ha="center",
    fontsize=9,
)
fig.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(visual_dir / "transport_sensitivity_analysis.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved analysis tables to:", output_dir)
print("Saved chart to:", visual_dir / "transport_sensitivity_analysis.png")

## Sources and limitations

### Sources
- Kenya National Bureau of Statistics (KNBS), [CPI Explorer](https://cpiexplorer.knbs.or.ke/)
- KNBS, [Consumer Price Indices and Inflation Rates — September 2026](https://www.knbs.or.ke/reports/consumer-price-indices-and-inflation-rates-september-2026/)
- Raw price file: `data/raw/knbs-prices.csv` in this repository

### Limitations to keep in mind
- The dataset contains 16 selected observations, not the complete official CPI basket.
- The fixed-quantity grocery basket is an illustrative comparison, not a monthly food bill.
- The rent and added energy reference prices may not match the prices faced by a particular household.
- Most household budget lines are assumptions and need validation against actual expenditure records.
- Transport fares are hypothetical scenarios, not verified national averages.
- Comparing two periods shows what changed in the selected observations; it does not explain why prices changed.

## Conclusion

The selected fixed-quantity grocery basket increased by 8.03% between September 2025 and September 2026. The transport scenarios show that higher assumed fares can leave much less income available under the household budget used here.

The useful lesson is not that this scenario represents every Kenyan household. It is that transparent assumptions, repeatable calculations and clear limitations help turn a small dataset into an analysis that can be reviewed and questioned.